# Project 7 — Stream Analysis (Algorithms for Massive Datasets, 2025/26)

Ho trattato la sequenza dei commenti NYT come uno stream (ordinato per `createDate`)
e ho implementato tre algoritmi:

1. **Flajolet-Martin** — n. di utenti distinti (`userID`)
2. **Alon-Matias-Szegedy (AMS)** — secondo momento per sezione
3. **Bloom filter** — test di appartenenza degli articoli a una sezione

Per FM e AMS confronto la stima con il valore esatto calcolato offline.

## Setup Ambiente, Colab o Locale

In [325]:
import sys
import os
import glob
import hashlib
import math
import time
import random
import numpy as np
import pandas as pd


try:
    import google.colab
    from google.colab import userdata
    # Su Colab: le credenziali Kaggle si leggono da *Secrets* (`KAGGLE_USERNAME`, `KAGGLE_KEY`).
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    !pip install -q kaggle
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(IN_COLAB)

False


In [326]:
# parametri globali
SUBSAMPLE = True       # False = dataset intero
SAMPLE_ROWS = 200000
SEED = 1

DATA_DIR = "dataset"
DATASET = "benjaminawd/new-york-times-articles-comments-2020"

# Flajolet-Martin
FM_NUM_HASHES = 64
FM_NUM_GROUPS = 8

# AMS
AMS_NUM_VARS = 100

# Bloom filter
BF_SECTION = "Opinion"
BF_TARGET_FPR = 0.01   # FPR target: m viene calcolato automaticamente da n
BF_K_HASHES = 7

## download dataset

In [327]:
def download_dataset():
    if os.path.isdir(DATA_DIR) and glob.glob(os.path.join(DATA_DIR, "nyt-comments-part*.csv")):
        print(f"Dataset gia' presente in '{DATA_DIR}'.")
        return
    
    os.makedirs(DATA_DIR, exist_ok=True)
    ret = os.system(f"kaggle datasets download -d {DATASET} -p {DATA_DIR} --unzip")
    if ret != 0:
        raise RuntimeError("Download fallito. Verificare le credenziali Kaggle.")

    print("Download completato.")


download_dataset()

Dataset gia' presente in 'dataset'.


In [328]:
part0 = os.path.join(DATA_DIR, "nyt-comments-part0.csv")

df = pd.read_csv(part0)
print(df.shape)      # righe, colonne
print(df.columns)    # nomi delle colonne
df.head(5)            # tipi e valori mancanti

(500000, 23)
Index(['commentID', 'status', 'commentSequence', 'userID', 'userDisplayName',
       'userLocation', 'userTitle', 'commentBody', 'createDate', 'updateDate',
       'approveDate', 'recommendations', 'replyCount', 'editorsSelection',
       'parentID', 'parentUserDisplayName', 'depth', 'commentType', 'trusted',
       'recommendedFlag', 'permID', 'isAnonymous', 'articleID'],
      dtype='object')


,commentID,status,commentSequence,userID,userDisplayName,userLocation,userTitle,commentBody,createDate,updateDate,...,editorsSelection,parentID,parentUserDisplayName,depth,commentType,trusted,recommendedFlag,permID,isAnonymous,articleID
0,104387472,approved,104387472,60215558,magicisnotreal,earth,NaN,Here is something I think is fraudulent that v...,2020-01-01 01:05:46,2020-01-01 08:13:39,...,False,NaN,NaN,1,comment,0,0,104387472,False,nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3...
1,104387873,approved,104387873,65691034,JD,Elko,NaN,@magicisnotreal I have used my VA loan option...,2020-01-01 01:52:25,2020-01-01 20:55:19,...,False,104387472.0,magicisnotreal,2,userReply,0,0,104387873,False,nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3...
2,104387976,approved,104387976,65110053,ebmem,"Memphis, TN",NaN,@magi\n\nWhy would someone take out a VA loan ...,2020-01-01 02:06:05,2020-01-01 20:55:35,...,False,104387472.0,magicisnotreal,2,userReply,0,0,104387976,False,nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3...
3,104390628,approved,104390628,60215558,magicisnotreal,earth,NaN,@JD\nOut here in the Alabama of the PNW they w...,2020-01-01 14:38:50,2020-01-01 20:56:46,...,False,104387873.0,magicisnotreal,2,userReply,0,0,104390628,False,nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3...
4,104391463,approved,104391463,65691034,JD,Elko,NaN,@magicisnotreal just a guess but I doubt that...,2020-01-01 16:23:14,2020-01-01 16:25:57,...,False,104390628.0,magicisnotreal,2,userReply,0,0,104391463,False,nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3...


## 2. Costruzione dello stream

I commenti vengono ordinati per `createDate` che ha il formato `YYYY-MM-DD HH:MM:SS` in modo da rispettare l'arrivo temporale.

i commenti verranno valutati uno alla volta tramite i cicli for. in uno stream reale ogni evento trigghera la relativa funzione update.

In [329]:
def load_comments(subsample=SUBSAMPLE, n=SAMPLE_ROWS):
    articles = pd.read_csv(
        os.path.join(DATA_DIR, "nyt-articles-2020.csv"),
        usecols=["uniqueID", "section"],
    )

    needed_cols = ["userID", "createDate", "articleID"]

    if subsample:
        df = pd.read_csv(os.path.join(DATA_DIR, "nyt-comments-part0.csv"), usecols=needed_cols)
        df = df.sort_values("createDate").head(n) # li riordino per simulare uno stream reale (prendo i primi 200.000)
    else:
        df = pd.read_csv(os.path.join(DATA_DIR, "nyt-comments-2020.csv"), usecols=needed_cols)
        df = df.sort_values("createDate")

    df = df.merge(articles, left_on="articleID", right_on="uniqueID", how="left") #merge tra uniqueID e articleID
    df = df.drop(columns="uniqueID")

    print(f"Stream: {len(df):,} commenti | {df['section'].nunique()} sezioni | subsample={subsample}")
    return df



In [330]:
df = load_comments()
df.head(3)

Stream: 200,000 commenti | 30 sezioni | subsample=True


,userID,createDate,articleID,section
0,42572458,2017-10-24 15:27:40,nyt://article/5e245bf7-e761-5b33-90e4-05f7e401...,The Learning Network
1,50727767,2018-01-12 18:18:57,nyt://article/5e245bf7-e761-5b33-90e4-05f7e401...,The Learning Network
2,125419,2019-01-22 02:37:48,nyt://article/5e245bf7-e761-5b33-90e4-05f7e401...,The Learning Network


In [331]:
print(df["section"].value_counts()) 

section
Opinion                 80972
U.S.                    39315
World                   37724
New York                 7198
Business Day             4800
Crosswords & Games       4162
The Learning Network     3675
Well                     2851
Health                   2249
Magazine                 2045
Movies                   1813
Style                    1794
Arts                     1647
The Upshot               1451
Sports                   1284
Climate                  1170
Real Estate              1134
Food                     1072
Technology                942
Travel                    717
Books                     599
Science                   439
Reader Center             310
Theater                   193
Obituaries                130
Podcasts                   90
The Weekly                 86
Fashion & Style            76
T Magazine                 45
Automobiles                17
Name: count, dtype: int64


## Flajolet-Martin (utenti distinti)

Se si hasha ogni elemento con una funzione uniforme su $\{0,1\}^b$, la probabilità
che il risultato termini con almeno $k$ zeri è $2^{-k}$.

In [332]:
_PRIME = (1 << 61) - 1

class FlajoletMartin:

    def __init__(self, num_hashes=FM_NUM_HASHES, num_groups=FM_NUM_GROUPS, seed=SEED):
        self.seed = seed
        rng = random.Random(seed)
        self.a = [rng.randint(1, _PRIME - 1) for _ in range(num_hashes)]
        self.b = [rng.randint(0, _PRIME - 1) for _ in range(num_hashes)]
        self.num_hashes = num_hashes
        self.num_groups = num_groups
        self.r_max = [0] * num_hashes # inizializzo a 0 il max di ogni hash [0,0,0,...]
    
    def _trailing_zeros(self, h):
        if h == 0:
            return 64
        return (h & -h).bit_length() - 1

    '''
    def _trailing_zeros(self, h_bin):
        count = 0
        for bit in reversed(h_bin):
            if bit == '0':
                count += 1
            else:
                break
        return count

    def update(self, user_id):
        for i in range(self.num_hashes):
            h_str = hashlib.md5(f"{user_id}_{self.seed + i}".encode()).hexdigest() #genero un hex di 32 caratteri 
            h_bin = bin(int(h_str, 16))[2:] # elimino i primi 2 caratteri perchè bin mi aggiunge 0b nella conversione
            r = self._trailing_zeros(h_bin)
            if r > self.r_max[i]:
                self.r_max[i] = r
    '''

    def update(self, user_id):
        x = int(user_id)
        for i in range(self.num_hashes):
            h = (self.a[i] * x + self.b[i]) % _PRIME
            r = self._trailing_zeros(h)
            if r > self.r_max[i]:
                self.r_max[i] = r
    
    def estimate(self):
        r_array = np.array(self.r_max)
        groups = np.array_split(r_array, self.num_groups)
        group_estimates = []
        for g in groups:
            median_r = float(np.median(g))
            estimate = 2.0 ** median_r
            group_estimates.append(estimate)
        return float(np.mean(group_estimates))


def exact_distinct_users(df):
    return df["userID"].nunique()

In [333]:
def run_fm(df):
    print("Flajolet-Martin")
    fm = FlajoletMartin()
    t0 = time.time()
    for user_id in df["userID"]:
        fm.update(user_id)
    elapsed = time.time() - t0

    est = fm.estimate()
    exact = exact_distinct_users(df)
    err = abs(est - exact) / exact
    print(f"  stima={est:,.0f}  esatto={exact:,}  errore={err:.1%}  ({elapsed:.1f}s)")
    return est, exact

In [334]:
def fm_sensitivity(df):
    print("FM: errore vs num_hashes")
    exact = exact_distinct_users(df)
    hash_counts=(8, 16, 32, 64, 128)
    #hash_counts=(256, 512)
    for num_h in hash_counts:
        fm = FlajoletMartin(num_hashes=num_h)
        for user_id in df["userID"]:
            fm.update(user_id)
        est = fm.estimate()
        err = abs(est - exact) / exact
        print(f"  H={num_h}: {est:,.0f}  (errore {err:.1%})")

In [335]:
run_fm(df)

Flajolet-Martin
  stima=53,042  esatto=50,876  errore=4.3%  (2.8s)


(53042.16563018069, 50876)

In [336]:
fm_sensitivity(df)

FM: errore vs num_hashes
  H=8: 266,240  (errore 423.3%)
  H=16: 69,426  (errore 36.5%)
  H=32: 96,281  (errore 89.2%)
  H=64: 53,042  (errore 4.3%)
  H=128: 48,449  (errore 4.8%)


## Alon-Matias-Szegedy (secondo momento delle sezioni)

Il secondo momento $F_2 = \sum_j m_j^2$ misura lo sbilanciamento della distribuzione
dei commenti per sezione: più è alto, più lo stream è concentrato su poche sezioni.

Si campionano `AMS_NUM_VARS` posizioni $p_i$ uniformemente in $[0, n)$.
Per ognuna si fissa l'elemento corrispondente e si conta quante volte riappare
nelle posizioni successive. Si dimostra che $\mathbb{E}[n(2v_i - 1)] = F_2$,
quindi la media delle variabili è uno stimatore non distorto.

In [337]:
class AMS:

    def __init__(self, n, num_vars=AMS_NUM_VARS, seed=SEED):
        rng = random.Random(seed)
        self.n = n
        self.num_vars = num_vars
        self.positions = sorted(rng.sample(range(n), num_vars))  
        self.elem = [None] * num_vars   
        self.count = [0] * num_vars     

    def update(self, idx, element):
        for i in range(self.num_vars):
            if self.positions[i] > idx:
                break                  # posizioni successive non ancora raggiunte
            if self.positions[i] == idx:
                self.elem[i] = element   # inizializzo la sezione
            if self.elem[i] == element:
                self.count[i] += 1       # aggiorno il conteggio comprendendo l'elemento appena inizializzato

    def estimate(self):
        vals = []
        for i in range(self.num_vars):
            v_i = self.count[i]                    
            vals.append(self.n * (2 * v_i - 1))   
        return float(np.mean(vals))


def exact_second_moment(df):
    freqs = df["section"].value_counts().to_numpy()
    return (freqs ** 2).sum()

In [338]:
def run_ams(df):
    print("AMS (secondo momento sezioni)")
    ams = AMS(len(df))
    t0 = time.time()
    idx = 0
    for section in df["section"]:
        ams.update(idx, section)
        idx += 1
    elapsed = time.time() - t0

    est = ams.estimate()
    exact = exact_second_moment(df)
    err = abs(est - exact) / exact
    print(f"  stima={est:,.0f}  esatto={exact:,}  errore={err:.1%}  ({elapsed:.1f}s)")
    return est, exact

In [339]:
def ams_sensitivity(df, var_counts=(10, 25, 50, 100, 200)):
    print("AMS: errore vs num_vars")
    exact = exact_second_moment(df)
    for num_v in var_counts:
        ams = AMS(n=len(df), num_vars=num_v)
        for idx, section in enumerate(df["section"]):
            ams.update(idx, section)
        est = ams.estimate()
        err = abs(est - exact) / exact
        print(f"  k={num_v}: {est:,.0f}  (errore {err:.1%})")

In [340]:
run_ams(df)

AMS (secondo momento sezioni)
  stima=9,214,792,000  esatto=9,667,184,406  errore=4.7%  (0.5s)


(9214792000.0, np.int64(9667184406))

In [341]:
ams_sensitivity(df)

AMS: errore vs num_vars
  k=10: 9,246,760,000  (errore 4.3%)
  k=25: 9,891,080,000  (errore 2.3%)
  k=50: 9,677,872,000  (errore 0.1%)
  k=100: 9,214,792,000  (errore 4.7%)
  k=200: 9,716,568,000  (errore 0.5%)


## Bloom filter (appartenenza a una sezione)

Inserimento: si settano a 1 i bit $h_1(x), \ldots, h_k(x)$.

Query: se anche solo un bit è 0, l'elemento è certamente assente (no falsi negativi).
I falsi positivi hanno probabilità teorica $p=(1 - e^{-kn/m})^k$.

Le funzioni hash sono $h_i(x) = \text{MD5}(x \| \text{salt}_i) \bmod m$.


In [342]:
class BloomFilter:

    def __init__(self, n, target_fpr=BF_TARGET_FPR, k_hashes=BF_K_HASHES, seed=SEED):
        rng = np.random.default_rng(seed)
        self.k = k_hashes
        # m per k fisso: m = -k*n / ln(1 - p^(1/k))
        self.m = math.ceil(-self.k * n / math.log(1 - target_fpr ** (1 / self.k))) #uso il ceil per arrotondare al bit superiore
        self.bits = np.zeros(self.m, dtype=bool)
        self.n_inserted = 0
        self.salts = []
        for _ in range(k_hashes):                          # k salt casuali, uno per funzione hash
            self.salts.append(str(rng.integers(0, 2**32)))

    def _positions(self, x):
        result = []
        for salt in self.salts:
            digest = hashlib.md5((x + salt).encode()).hexdigest()
            result.append(int(digest, 16) % self.m)
        return result

    def add(self, x):
        for pos in self._positions(x):
            self.bits[pos] = True
        self.n_inserted += 1

    def __contains__(self, x):
        for pos in self._positions(x):
            if not self.bits[pos]:
                return False
        return True

    def fpr_theoretical(self):
        if self.n_inserted == 0:
            return 0.0
        return (1.0 - math.exp(-self.k * self.n_inserted / self.m)) ** self.k

In [343]:
def run_bloom(df):
    print(f"Bloom filter (sezione: '{BF_SECTION}')")

    articles = df.drop_duplicates("articleID")[["articleID", "section"]]
    in_section  = articles[articles["section"] == BF_SECTION]["articleID"].tolist()
    out_section = articles[articles["section"] != BF_SECTION]["articleID"].tolist()

    bf = BloomFilter(n=len(in_section))
    for art_id in in_section:
        bf.add(art_id)

    fp = sum(1 for art in out_section if art in bf)
    fpr_emp = fp / len(out_section)
    fpr_theo = bf.fpr_theoretical()

    print(f"  inseriti={len(in_section)}  non-{BF_SECTION}={len(out_section)}")
    print(f"  FP={fp}  FPR empirica={fpr_emp:.2%}  FPR teorica={fpr_theo:.2%}")
    print(f"  m={bf.m:,} bit  k={BF_K_HASHES}  target={BF_TARGET_FPR:.0%}")
    return fpr_emp, fpr_theo

In [344]:
def bloom_fpr_sensitivity(df, fpr_targets=(0.50, 0.20, 0.10, 0.05, 0.01)):
    print(f"Bloom FPR empirica vs teorica — sezione '{BF_SECTION}'")

    articles = df.drop_duplicates("articleID")[["articleID", "section"]]
    in_section  = articles[articles["section"] == BF_SECTION]["articleID"].tolist()
    out_section = articles[articles["section"] != BF_SECTION]["articleID"].tolist()

    for target_fpr in fpr_targets:
        bf = BloomFilter(n=len(in_section), target_fpr=target_fpr, k_hashes=BF_K_HASHES)
        for art_id in in_section:
            bf.add(art_id)

        fp = sum(1 for art in out_section if art in bf)
        fpr_emp = fp / len(out_section)
        fpr_theo = bf.fpr_theoretical()

        print(f"  target={target_fpr:.0%}  m={bf.m:,} bit - inseriti={bf.n_inserted}")
        print(f"    FP={fp}")
        print(f"    FPR empirica={fpr_emp:.2%}  FPR teorica={fpr_theo:.2%}")
        print()

    print(f"  ({len(out_section):,} articoli negativi testati, k={BF_K_HASHES})")

In [345]:
run_bloom(df)

Bloom filter (sezione: 'Opinion')
  inseriti=114  non-Opinion=570
  FP=4  FPR empirica=0.70%  FPR teorica=1.00%
  m=1,094 bit  k=7  target=1%


(0.007017543859649123, 0.009982491571523647)

In [346]:
bloom_fpr_sensitivity(df)

Bloom FPR empirica vs teorica — sezione 'Opinion'
  target=50%  m=338 bit - inseriti=114
    FP=343
    FPR empirica=60.18%  FPR teorica=49.98%

  target=20%  m=505 bit - inseriti=114
    FP=110
    FPR empirica=19.30%  FPR teorica=19.91%

  target=10%  m=628 bit - inseriti=114
    FP=55
    FPR empirica=9.65%  FPR teorica=9.97%

  target=5%  m=757 bit - inseriti=114
    FP=28
    FPR empirica=4.91%  FPR teorica=4.98%

  target=1%  m=1,094 bit - inseriti=114
    FP=4
    FPR empirica=0.70%  FPR teorica=1.00%

  (570 articoli negativi testati, k=7)
